# **Class Imbalance: Oversampling vs Undersampling**


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("smart_grid_stability_augmented.csv")
print(df.shape)
df['stabf'].value_counts()

(60000, 14)


stabf
unstable    38280
stable      21720
Name: count, dtype: int64

        The original `stabf` target is 64% unstable / 36% stable — only mildly imbalanced.
        That's not severe enough to show the real failure mode of imbalanced learning.

        To make a clear, downsample the `stable` class
        to simulate a dataset where it's rare (~4% of the data) — similar to fraud detection
        or disease diagnosis scenarios where the class we care about is naturally rare.

In [2]:
stable_df = df[df['stabf'] == 'stable']
unstable_df = df[df['stabf'] == 'unstable']

# Keep only 1500 'stable' rows out of ~21,720 -> simulates a rare minority class
stable_small = stable_df.sample(n=1500, random_state=42)

df_imb = pd.concat([unstable_df, stable_small]).sample(frac=1, random_state=42).reset_index(drop=True)

print(df_imb['stabf'].value_counts())
print(df_imb['stabf'].value_counts(normalize=True).round(3))

stabf
unstable    38280
stable       1500
Name: count, dtype: int64
stabf
unstable    0.962
stable      0.038
Name: proportion, dtype: float64



    `stratify=y` ensures both train and test sets keep the same ~96/4 imbalance ratio
    as the full dataset — this matters because the test set must reflect reality.

In [3]:
from sklearn.model_selection import train_test_split

X = df_imb.drop(columns=['stab', 'stabf'])   # drop leak-y continuous target too
y = df_imb['stabf']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

print("Train class counts:")
print(y_train.value_counts())
print("\nTest class counts:")
print(y_test.value_counts())

Train class counts:
stabf
unstable    28710
stable       1125
Name: count, dtype: int64

Test class counts:
stabf
unstable    9570
stable       375
Name: count, dtype: int64


    Logistic Regression (our model for this comparison) is sensitive to feature scale.

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train only
X_test_scaled = scaler.transform(X_test)          # never fit on test data

    Reuse this for every resampling technique, so each one gets compared fairly
    on the SAME untouched test set.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

def train_and_evaluate(name, X_resampled, y_resampled):
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_resampled, y_resampled)
    y_pred = model.predict(X_test_scaled)

    print(f"=== {name} ===")
    print("Training class counts:", pd.Series(y_resampled).value_counts().to_dict())
    print(classification_report(y_test, y_pred))

    return f1_score(y_test, y_pred, pos_label='stable')

    This shows the real danger of class imbalance. Watch the `stable` row closely:
    high precision, but very low recall. The model is barely catching any real
    "stable" cases — it's biased toward predicting the majority class.

In [6]:
f1_baseline = train_and_evaluate("Baseline (no resampling)", X_train_scaled, y_train)

=== Baseline (no resampling) ===
Training class counts: {'unstable': 28710, 'stable': 1125}
              precision    recall  f1-score   support

      stable       0.94      0.21      0.34       375
    unstable       0.97      1.00      0.98      9570

    accuracy                           0.97      9945
   macro avg       0.96      0.61      0.66      9945
weighted avg       0.97      0.97      0.96      9945



## Oversampling: Random Oversampling

    Duplicates random rows from the `stable` class until both classes have equal counts
    (28,710 each). Watch what happens to recall vs precision compared to baseline.

In [7]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(X_train_scaled, y_train)

f1_random_over = train_and_evaluate("Random Oversampling", X_res, y_res)

=== Random Oversampling ===
Training class counts: {'unstable': 28710, 'stable': 28710}
              precision    recall  f1-score   support

      stable       0.13      0.79      0.22       375
    unstable       0.99      0.79      0.88      9570

    accuracy                           0.79      9945
   macro avg       0.56      0.79      0.55      9945
weighted avg       0.96      0.79      0.85      9945



## Oversampling: SMOTE

    Instead of duplicating, SMOTE generates NEW synthetic `stable` points by interpolating
    between existing minority neighbors.

In [8]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train_scaled, y_train)

f1_smote = train_and_evaluate("SMOTE", X_res, y_res)

=== SMOTE ===
Training class counts: {'unstable': 28710, 'stable': 28710}
              precision    recall  f1-score   support

      stable       0.14      0.79      0.23       375
    unstable       0.99      0.80      0.89      9570

    accuracy                           0.80      9945
   macro avg       0.56      0.80      0.56      9945
weighted avg       0.96      0.80      0.86      9945



## Oversampling: ADASYN

    A refinement of SMOTE — generates more synthetic points in regions where the
    minority class is hardest to learn (near the decision boundary), fewer where
    it's already easy.

In [9]:
from imblearn.over_sampling import ADASYN

adasyn = ADASYN(random_state=42)
X_res, y_res = adasyn.fit_resample(X_train_scaled, y_train)

f1_adasyn = train_and_evaluate("ADASYN", X_res, y_res)

=== ADASYN ===
Training class counts: {'unstable': 28710, 'stable': 28701}
              precision    recall  f1-score   support

      stable       0.13      0.79      0.22       375
    unstable       0.99      0.79      0.88      9570

    accuracy                           0.79      9945
   macro avg       0.56      0.79      0.55      9945
weighted avg       0.96      0.79      0.86      9945



## Undersampling: Random Undersampling

        Drops random rows from the majority (`unstable`) class until it matches the
        minority class size. Notice the training set shrinks dramatically (29,835 -> 2,250 rows).

In [10]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_res, y_res = rus.fit_resample(X_train_scaled, y_train)

f1_random_under = train_and_evaluate("Random Undersampling", X_res, y_res)

=== Random Undersampling ===
Training class counts: {'stable': 1125, 'unstable': 1125}
              precision    recall  f1-score   support

      stable       0.13      0.80      0.23       375
    unstable       0.99      0.79      0.88      9570

    accuracy                           0.79      9945
   macro avg       0.56      0.80      0.55      9945
weighted avg       0.96      0.79      0.86      9945



## Undersampling: ClusterCentroids

    Clusters the majority class with K-Means and replaces each cluster with its
    centroid (a synthetic representative point) instead of picking real rows.
    This is noticeably slower — it has to run K-Means internally.

In [11]:
from imblearn.under_sampling import ClusterCentroids
import time

t0 = time.time()
cc = ClusterCentroids(random_state=42)
X_res, y_res = cc.fit_resample(X_train_scaled, y_train)
print(f"ClusterCentroids took {time.time()-t0:.1f}s")

f1_cluster_centroids = train_and_evaluate("ClusterCentroids", X_res, y_res)

ClusterCentroids took 3.9s
=== ClusterCentroids ===
Training class counts: {'stable': 1125, 'unstable': 1125}
              precision    recall  f1-score   support

      stable       0.13      0.81      0.22       375
    unstable       0.99      0.78      0.87      9570

    accuracy                           0.78      9945
   macro avg       0.56      0.79      0.54      9945
weighted avg       0.96      0.78      0.85      9945



## Undersampling: NearMiss

    Instead of dropping majority points randomly, NearMiss selectively KEEPS the
    majority points that are closest to the minority class (near the decision
    boundary) and discards the "easy", far-away majority points.

In [12]:
from imblearn.under_sampling import NearMiss

nm = NearMiss(version=1, n_neighbors=3)
X_res, y_res = nm.fit_resample(X_train_scaled, y_train)

f1_nearmiss = train_and_evaluate("NearMiss", X_res, y_res)

=== NearMiss ===
Training class counts: {'stable': 1125, 'unstable': 1125}
              precision    recall  f1-score   support

      stable       0.22      0.66      0.33       375
    unstable       0.99      0.91      0.95      9570

    accuracy                           0.90      9945
   macro avg       0.60      0.78      0.64      9945
weighted avg       0.96      0.90      0.92      9945



##  Side-by-side comparison

    All methods evaluated on the exact same untouched test set, so the comparison is fair.

In [13]:
comparison = pd.DataFrame({
    'Method': ['Baseline (no resampling)', 'Random Oversampling', 'SMOTE', 'ADASYN',
               'Random Undersampling', 'ClusterCentroids', 'NearMiss'],
    'Minority (stable) F1': [f1_baseline, f1_random_over, f1_smote, f1_adasyn,
                              f1_random_under, f1_cluster_centroids, f1_nearmiss]
}).sort_values('Minority (stable) F1', ascending=False).reset_index(drop=True)

comparison

,Method,Minority (stable) F1
0,Baseline (no resampling),0.344227
1,NearMiss,0.329538
2,SMOTE,0.230619
3,Random Undersampling,0.226258
4,ADASYN,0.223645
5,Random Oversampling,0.223388
6,ClusterCentroids,0.217143


## What actually happened here? 

  This is an honest, slightly counter-intuitive result — and it's a more realistic
  lesson than "resampling always helps":

  - **Baseline** had the highest minority F1 (~0.34) but terrible recall (~0.21) —
    it correctly identifies very few `stable` cases, but when it does predict
    `stable`, it's usually right (high precision).
  - **All the oversampling methods (Random, SMOTE, ADASYN)** dramatically improved
    recall (21% -> ~79%) but precision collapsed (94% -> ~13%). The model now
    catches almost all real `stable` cases, but also wrongly flags many `unstable`
    cases as `stable`. Net F1 actually went DOWN slightly compared to baseline.
  - **NearMiss** struck the best balance among the resampling techniques — better
    recall than baseline (66% vs 21%) while keeping precision reasonable (22%),
    giving it the best F1 among all the resampling methods tried.

  **The real lesson:** resampling is not a magic fix — it shifts the tradeoff
  between precision and recall. Whether that tradeoff is "better" depends entirely
  on your use case:

  | Use case | What matters more | What to prefer |
  |---|---|---|
  | Fraud detection | Catching as much fraud as possible (recall) | Oversampling / NearMiss |
  | Spam filter | Not blocking real emails (precision) | Baseline or mild techniques |
  | Medical screening | Recall (missing a disease is costly) | Oversampling |
  | Recommendation flagging | Precision (false positives erode trust) | Baseline / undersampling carefully |

  **Always look at precision AND recall, not just F1 or accuracy**, and pick the
  resampling strategy that matches what mistake is more costly in your specific
  problem.

## A cheaper alternative worth trying first: `class_weight='balanced'`

    Before reaching for resampling, try this — it costs nothing in terms of data
    manipulation, training time, or synthetic data risk. It just tells the model
    to penalize minority misclassifications more heavily during training.

In [14]:
model_weighted = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model_weighted.fit(X_train_scaled, y_train)
y_pred_weighted = model_weighted.predict(X_test_scaled)

from sklearn.metrics import classification_report
print("=== class_weight='balanced' (no resampling at all) ===")
print(classification_report(y_test, y_pred_weighted))

f1_weighted = f1_score(y_test, y_pred_weighted, pos_label='stable')
print(f"Minority F1: {f1_weighted:.3f}")

=== class_weight='balanced' (no resampling at all) ===
              precision    recall  f1-score   support

      stable       0.13      0.80      0.23       375
    unstable       0.99      0.79      0.88      9570

    accuracy                           0.79      9945
   macro avg       0.56      0.80      0.55      9945
weighted avg       0.96      0.79      0.86      9945

Minority F1: 0.225


    `class_weight='balanced'` alone gets you most of the benefit of resampling
    with none of the added complexity or training time. It's almost always worth
    trying first.